# 03 — Fitting the implied volatility surface with SVI

We take the committed sample chain, clean it, invert each price to an implied
vol, and fit one SVI slice per expiry:

    w(k) = a + b ( rho (k - m) + sqrt((k - m)^2 + sigma_svi^2) ),   k = ln(K/F)

Fit in **total variance** `w = sigma^2 T`, not in vol: that is the space where
arbitrage-free interpolation is (nearly) linear, and where the wings behave.

Then we check the two static no-arbitrage conditions:

* **butterfly**: Gatheral's g(k) >= 0 for every k (risk-neutral density >= 0);
* **calendar**: total variance must not decrease with maturity at fixed k.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from optlab.pricing.black_scholes import bs_price, bs_greeks
from optlab.pricing.binomial import binomial_price
from optlab.pricing.monte_carlo import mc_price

from optlab.data.cleaning import clean_chain
from optlab.data.loaders import load_csv_chain
from optlab.surface.arbitrage import check_surface, is_arbitrage_free, scan_butterfly
from optlab.surface.interpolate import VolSurface

## 1. Load and clean

`clean_chain` drops the quotes a fit cannot trust and attaches `T`, the parity
implied `forward`, log-moneyness `k` and `iv`. The report tells you exactly what
was removed — always look at it before blaming the model for a bad fit.
Note the delta-band line in that report. Without it, the 31-day slice fits to 3.4
vol points and reports a butterfly arbitrage: the far wings are penny-priced
options whose vega is too small to carry information. Filtering by delta instead of
a fixed strike range adapts to expiry and vol level automatically — always read the
report before blaming the fit.

In [ ]:
raw = load_csv_chain("data/sample/option_chain_sample.csv")
spot = raw.attrs["spot"]

chain, report = clean_chain(raw, spot=spot, r=0.04, q=0.015, return_report=True)
print(report.summary())
chain[["expiry", "strike", "kind", "mid", "T", "forward", "k", "iv"]].head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for T, grp in chain.groupby("T"):
    calls = grp[grp["kind"] == "call"].sort_values("strike")
    ax.plot(calls["strike"] / calls["forward"], calls["iv"] * 100, "o-", markersize=3, label=f"T={T:.2f}")
ax.set_xlabel("strike / forward")
ax.set_ylabel("implied vol (%)")
ax.set_title("The smile: implied vol by moneyness and expiry")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Fit SVI per expiry and check quality

The RMSE is reported in **vol points**. A good fit is a few tenths of a point;
much more than that means the quotes are noisy or the slice is not really SVI-shaped
(e.g. an earnings event inside the expiry, which breaks the single-slice assumption).

In [ ]:
surface = VolSurface.from_quotes(chain["strike"], chain["iv"], chain["T"], chain["forward"])
print(surface)
surface.diagnostics().round(5)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey=False)
for ax, T in zip(axes.ravel(), surface.maturities):
    grp = chain[np.isclose(chain["T"], T)]
    fwd = float(grp["forward"].iloc[0])
    k_quotes = grp["k"].to_numpy()
    ax.plot(k_quotes, grp["iv"] * 100, "o", markersize=3, label="quotes")

    k_grid = np.linspace(k_quotes.min() - 0.05, k_quotes.max() + 0.05, 100)
    idx = int(np.argmin(np.abs(surface.maturities - T)))
    ax.plot(k_grid, surface.params[idx].iv(k_grid, T) * 100, "-", label="SVI fit")

    ax.set_title(f"T = {T:.2f}y  (F = {fwd:.2f})")
    ax.set_xlabel("ln(K/F)")
    ax.set_ylabel("implied vol (%)")
    ax.legend(fontsize=7)
for ax in axes.ravel()[len(surface.maturities):]:
    ax.axis("off")
fig.suptitle("SVI slices fitted to the sample chain")
fig.tight_layout()
plt.show()

## 3. Arbitrage checks

Run every butterfly check plus each adjacent calendar check. The numbers below
are the *worst* value found on the scan grid; negative would mean free money.

In [ ]:
reports = check_surface(surface.params, list(surface.maturities), k_min=-0.6, k_max=0.4)
for rep in reports:
    print(rep.summary())
print("\nsurface arbitrage-free:", is_arbitrage_free(reports))

## 4. Reading the surface anywhere

A `VolSurface` answers "what vol for this strike and expiry?" for any `(K, T)` in
between: SVI across strikes, linear in total variance across maturities, flat vol
outside the fitted range. Hold wings fixed and interpolate *variance*, never vol —
interpolating vol can create a calendar arbitrage out of nowhere.

In [ ]:
strikes = np.array([90.0, 95.0, 100.0, 105.0, 110.0])
maturities = np.array([0.15, 0.30, 0.60, 1.20])
grid = surface.iv_grid(strikes, maturities)

pd.DataFrame(grid * 100, index=[f"T={t:.2f}" for t in maturities],
             columns=[f"K={k:.0f}" for k in strikes]).round(2)

In [ ]:
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")
K_grid = np.linspace(80, 125, 31)
T_grid = np.linspace(0.09, 1.0, 26)
XX, TT = np.meshgrid(K_grid, T_grid)
ZZ = surface.iv_grid(K_grid, T_grid) * 100
ax.plot_surface(XX, TT, ZZ, cmap="viridis", edgecolor="none", alpha=0.9)
ax.set_xlabel("strike")
ax.set_ylabel("maturity (years)")
ax.set_zlabel("implied vol (%)")
ax.set_title("Interpolated volatility surface")
plt.tight_layout()
plt.show()

## Takeaway

The surface is not one number: it is a smooth function with a skew that steepens
as expiry shortens, and it must be arbitrage-free to be usable for pricing. SVI
gives you five parameters per expiry you can reason about (`rho` is the skew,
`b` the wing slope, `sigma_svi` the smile curvature), and the checks tell you when
a fit has drifted into fantasy.

Next: `04_delta_hedging_pnl.ipynb` asks what happens when you trade at these vols.